In [2]:
pip install langchain langchain-community langchain-mistralai chromadb pypdf tiktoken mistralai -U


In [20]:
pip install langchain-chroma -U

In [ ]:
# After installing the new package, modify the import and initialization
from langchain_mistralai import MistralAIEmbeddings
from langchain_chroma import Chroma

In [22]:
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import Chroma


In [14]:
from langchain_core.documents import Document

In [23]:
documents = [
    Document(
        page_content="Virat Kohli is an Indian international cricketer who plays for Royal Challengers Bangalore in the IPL and formerly captained the Indian national cricket team. Known as the ""King Kohli"", he is widely regarded as one of the greatest batsmen of all time.",
        metadata={
            "source": "wikipedia",
            "player": "virat kohli",
            "team": "RCB"
        }
    ),
    Document(
        page_content="Rohit Sharma is an Indian international cricketer who plays for Mumbai Indians in the IPL and is the current captain of the Indian national cricket team in all formats. Known for his elegant batting style and leadership.",
        metadata={
            "source": "wikipedia",
            "player": "rohit sharma",
            "team": "Mumbai Indians"
        }
    ),
    Document(
        page_content="Mahendra Singh Dhoni, commonly known as MS Dhoni, is an Indian professional cricketer who played as a wicket-keeper-batsman. He captained the Indian national team in limited-overs formats from 2007 to 2017 and in Test cricket from 2008 to 2014. He led India to victory in the 2007 ICC World Twenty20, the 2010 and 2016 Asia Cups, the 2011 Cricket World Cup and the 2013 ICC Champions Trophy.",
        metadata={
            "source": "wikipedia",
            "player": "ms dhoni",
            "team": "CSK"
        }
    )
]

print(f"Created {len(documents)} documents.")

Created 3 documents.


In [24]:
new_documents = [
    Document(
        page_content="Sachin Tendulkar is a former Indian international cricketer who captained the Indian national team. Widely regarded as one of the greatest batsmen in the history of cricket, he is the highest run-scorer of all time in International cricket.",
        metadata={
            "source": "wikipedia",
            "player": "sachin tendulkar",
            "team": "Mumbai Indians"
        }
    ),
    Document(
        page_content="Rahul Dravid is a former Indian cricketer and captain who is currently the head coach of the Indian national cricket team. Known as 'The Wall' for his resilient batting, he is one of only four batsmen to score 10,000 runs in both Tests and ODIs.",
        metadata={
            "source": "wikipedia",
            "player": "rahul dravid",
            "team": "Rajasthan Royals"
        }
    ),
    Document(
        page_content="Sourav Ganguly, often affectionately known as 'Dada', is a former Indian cricketer, commentator and administrator who captained the Indian national team. He is widely regarded as one of India's most successful captains in Test cricket.",
        metadata={
            "source": "wikipedia",
            "player": "sourav ganguly",
            "team": "Kolkata Knight Riders"
        }
    )
]

documents.extend(new_documents)

print(f"Added {len(new_documents)} more documents. Total documents: {len(documents)}")

Added 3 more documents. Total documents: 6


In [28]:
import os
from google.colab import userdata
try:
    # Ensure your MISTRAL_API_KEY is set in Colab secrets
    os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

    # Initialize the vector store using the documents created earlier
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=MistralAIEmbeddings(),
        persist_directory="chroma_db",
        collection_name="cricket_players"
    )
    print("Vector store created successfully!")
except userdata.SecretNotFoundError:
    print("Error: MISTRAL_API_KEY not found in Secrets.")
    print("Please add it to the Secrets tab (key icon) and enable notebook access.")

Vector store created successfully!


# New section

In [31]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['3c85bf9d-cbce-4eda-be3c-13e31eda5a83',
  'b681a951-40ef-4046-ada0-669649db78b6',
  'eec4c120-bba1-403d-82a0-493a89f4f17c',
  '32a30db8-04c1-42e8-900f-f1c125399873',
  'bc58c4d1-5730-4404-8ff4-362d64735a1f',
  '26c360db-f41a-43c5-8cb4-d35ade100b09'],
 'embeddings': array([[-0.03649902,  0.00403595,  0.04464722, ..., -0.01154327,
          0.04873657, -0.01161194],
        [-0.04043579,  0.02359009,  0.06646729, ..., -0.01125336,
          0.05224609, -0.02767944],
        [-0.03451538,  0.01625061,  0.02740479, ..., -0.01615906,
          0.04711914,  0.00079918],
        [-0.02055359,  0.00792694,  0.01350403, ..., -0.00107479,
          0.03062439, -0.01800537],
        [-0.03997803,  0.00904083,  0.03045654, ..., -0.01733398,
          0.04650879, -0.00458908],
        [-0.05307007, -0.00408554,  0.03050232, ...,  0.00305939,
          0.03720093, -0.00536728]]),
 'documents': ['Virat Kohli is an Indian international cricketer who plays for Royal Challengers Bangalore in th

In [34]:
vector_store.similarity_search(
    query="who  among these are batsmap who become coach of india",
    k=1 #number of result
)

[Document(metadata={'team': 'Rajasthan Royals', 'player': 'rahul dravid', 'source': 'wikipedia'}, page_content="Rahul Dravid is a former Indian cricketer and captain who is currently the head coach of the Indian national cricket team. Known as 'The Wall' for his resilient batting, he is one of only four batsmen to score 10,000 runs in both Tests and ODIs.")]

In [35]:
vector_store.similarity_search_with_score(
    query="who  among these are batsmap who become coach of india",
    k=1 #number of result
)

[(Document(metadata={'team': 'Rajasthan Royals', 'source': 'wikipedia', 'player': 'rahul dravid'}, page_content="Rahul Dravid is a former Indian cricketer and captain who is currently the head coach of the Indian national cricket team. Known as 'The Wall' for his resilient batting, he is one of only four batsmen to score 10,000 runs in both Tests and ODIs."),
  0.43613797426223755)]

In [37]:
vector_store.similarity_search_with_score(
    query="",
    filter={
        "team": "Mumbai Indians"
    },
   #number of result
)

[(Document(metadata={'player': 'sachin tendulkar', 'source': 'wikipedia', 'team': 'Mumbai Indians'}, page_content='Sachin Tendulkar is a former Indian international cricketer who captained the Indian national team. Widely regarded as one of the greatest batsmen in the history of cricket, he is the highest run-scorer of all time in International cricket.'),
  0.6327372789382935),
 (Document(metadata={'source': 'wikipedia', 'team': 'Mumbai Indians', 'player': 'rohit sharma'}, page_content='Rohit Sharma is an Indian international cricketer who plays for Mumbai Indians in the IPL and is the current captain of the Indian national cricket team in all formats. Known for his elegant batting style and leadership.'),
  0.656180739402771)]